In [0]:
%run ../delta_function

In [0]:
import os
import pandas as pd

from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql import Window
import pyspark.sql.utils;
from pyspark.sql.types import StructType, StringType, FloatType, IntegerType, TimestampType
from pyspark.sql.functions import concat, lit, col, upper, max, when, row_number, length, to_date, udf, current_timestamp, regexp_replace, date_format, first, expr
from datetime import datetime, timedelta
import numpy as np
from pyspark.sql import SparkSession
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [0]:
try:
    verbose_mode = dbutils.widgets.get("verbose_mode");
except:
    verbose_mode = 'debug'
try:
    current_division = dbutils.widgets.get("division");
except:
    current_division = 'mal'
try:
    current_environment = dbutils.widgets.get("environment");
except:
    current_environment = 'dev'
try:
    execution_mode = dbutils.widgets.get("execution_mode");
except:
    execution_mode = 'update'
try:
    current_project = dbutils.widgets.get("project");
except:
    current_project = 'maite_bi'
try:
    current_production_line = dbutils.widgets.get("production_line");
except:
    current_production_line = 'gains'

current_catalog = current_division + '_' + current_project + '_' + current_environment;

current_schema = current_production_line;

current_location = 'abfss://' + current_project + '@adlsdpcom'+ current_environment +f'data.dfs.core.windows.net/' + current_production_line + '/'

if verbose_mode == 'debug':
    display("Debug Mode")
    display(f"current_division : {current_division}")
    display(f"current_environment : {current_environment}")
    display(f"current_project : {current_project}")
    display(f"current_production_line : {current_production_line}")
    display(f"current_catalog : {current_catalog}")
    display(f"current_schema : {current_schema}")
    display(f"current_location : {current_location}")

Source(s)

In [0]:
source_temp = f"""mal_maite_bi_{current_environment}.gains"""

In [0]:
source_gold = f"""mal_maite_common_{current_environment}.gold"""

In [0]:
source_mal_maite = f"""mal_maite_{current_environment}"""

In [0]:
if current_environment =='preprd':
    source = f"""ext_mal_psql_maite_vision_board_test.public"""    
else:
    source = f"""ext_mal_psql_maite_vision_board_{current_environment}.public"""


Fact

In [0]:
batches = f"""
SELECT b.id_batch, b.fabrication_order_number, b.batch_number, b.mes_number, b.production_line AS id_plant, ppl.name AS production_line, b.requirement_specifications AS id_cahier_des_charges, rs.name AS specifications_name, ppt.code AS production_type, gs.code AS goods_specy, gv.code AS goods_variety, b.harvest AS goods_harvest, pv.code AS parameter, me.value
FROM {source}.batches b

LEFT JOIN {source}.plants_production_lines ppl ON b.production_line = ppl.id_plant_production_line
LEFT JOIN {source}.requirement_specifications rs ON rs.id_requirement_specification = b.requirement_specifications AND rs.deleted = false
LEFT JOIN {source}.parameters_production_types ppt ON ppt.id_parameter_production_type = b.production_type AND ppt.deleted = false
LEFT JOIN {source}.goods_varieties gv ON gv.id_good_variety = b.variety
LEFT JOIN {source}.goods_species gs ON gs.id_good_specy = gv.specy

LEFT JOIN {source}.manual_entries me ON me.batch = b.id_batch AND me.deleted = false
JOIN {source}.parameters_variables pv ON pv.id_parameter_variable = me.parameter

WHERE b.deleted = false AND b.planned_date >= '2024-01-01' AND pv.code IN ('goods_moisture', 'goods_weight', 'malt_weight', 'malt_yield_r2')
"""

df_batches = spark.sql(batches)
df_batches.createOrReplaceTempView("batches")

# Méthode de définition des dates de fin de production
date_fin_production = f"""
SELECT b.id_batch, b.planned_date, 
        l.kiln_unload_end_date, 
        l.kiln_unload_start_date, 
        l.germ_unload_start_date, 
        l.steep_c1_o1_filling_start_date
FROM {source}.batches b
LEFT JOIN {source_gold}.localizations_pivot l ON b.id_batch = l.batch_id
"""

df_date_fin_production = spark.sql(date_fin_production)

# Calcul du champ fin_de_production selon la priorité
df_date_fin_production = df_date_fin_production.withColumn(
    "date_fin_production",
    when(col("kiln_unload_end_date").isNotNull(), col("kiln_unload_end_date"))
    .when(col("kiln_unload_start_date").isNotNull(), col("kiln_unload_start_date"))
    .when(col("germ_unload_start_date").isNotNull(), col("germ_unload_start_date") + expr("INTERVAL 2 DAYS"))
    .when(col("steep_c1_o1_filling_start_date").isNotNull(), col("steep_c1_o1_filling_start_date") + expr("INTERVAL 8 DAYS"))
    .otherwise(col("planned_date") + expr("INTERVAL 8 DAYS")))

df_date_fin_production.createOrReplaceTempView("date_fin_production")

# Jointure pour ajouter les dates de fin de production
main = f"""
SELECT b.*, dfp.date_fin_production
FROM batches b
LEFT JOIN date_fin_production dfp ON b.id_batch = dfp.id_batch
"""
df_main = spark.sql(main)

# Pivot
df_main = df_main.groupBy('id_batch', 'fabrication_order_number', 'batch_number', 'mes_number', 'date_fin_production', 'id_plant', 'production_line', 'id_cahier_des_charges', 'specifications_name', 'production_type', 'goods_specy', 'goods_variety', 'goods_harvest').pivot("parameter").agg(F.first("value"))

# Ajout du malt_moisture via table gold
df_main.createOrReplaceTempView("main")
main_gold = f"""
SELECT m.*, mqr.malt_moisture
FROM main m
LEFT JOIN {source_gold}.malt_quality_results mqr ON mqr.fabrication_order = m.fabrication_order_number
"""
df_main = spark.sql(main_gold)

In [0]:
# Renommer les colonnes baseline
df_main = df_main.withColumnRenamed("batch_number", "batch")
df_main = df_main.withColumnRenamed("fabrication_order_number", "ordre_de_fabrication")
df_main = df_main.withColumnRenamed("mes_number", "lot_mes")
df_main = df_main.withColumnRenamed("goods_harvest", "annee_de_recolte")
df_main = df_main.withColumnRenamed("id_plant", "id_site_production")
df_main = df_main.withColumnRenamed("production_line", "site_production")
df_main = df_main.withColumnRenamed("specifications_name", "cahier_des_charges")
df_main = df_main.withColumnRenamed("production_type", "type_de_production")
df_main = df_main.withColumnRenamed("goods_moisture", "reel_humidite_orge")
df_main = df_main.withColumnRenamed("goods_weight", "reel_orge_humide")
df_main = df_main.withColumnRenamed("malt_moisture", "reel_humidite_malt")
df_main = df_main.withColumnRenamed("malt_weight", "reel_malt_humide")
df_main = df_main.withColumnRenamed("malt_yield_r2", "reel_r2_humide")

In [0]:
# Trier le dataframe par la colonne batch
df_main = df_main.orderBy("date_fin_production")

# Supprime les batch avant le 15/01/2024 pour ROUEN1
df_main = df_main.filter((col("site_production") != "ROUEN1") | (col("date_fin_production") >= "2024-01-15"))

In [0]:
# Mise à jour des valeurs manquantes
df_main = (
    df_main
    .withColumn(
        'reel_orge_humide',
        when(col('batch') == 'C23F5508', 590)
        .when(col('batch') == 'C24F5701', 585)
        .otherwise(col('reel_orge_humide'))
    )
    .withColumn(
        'reel_humidite_orge',
        when(col('batch') == 'C23F5508', 11.5).otherwise(col('reel_humidite_orge'))
    )
    .withColumn(
        'reel_malt_humide',
        when(col('batch') == 'C24F5607', 502.427)
        .when(col('batch') == 'NG1_2024_7810', 189.199)
        .otherwise(col('reel_malt_humide'))
    )
    .withColumn(
        'reel_humidite_malt',
        when(col('batch') == 'C24F5607', 4.6)
        .when(col('batch') == 'C24F5613', 4.2)
        .when(col('batch') == 'C23N439', 4.3)
        .when(col('batch') == 'PR1_2024_3303', 4.5)
        .when(col('batch') == 'PR1_2024_3309', 4.4)
        .when(col('batch') == 'PR1_2024_3310', 4.4)
        .when(col('batch') == 'PR1_2024_3311', 4.3)
        .when(col('batch') == 'NG1_2024_7802', 4.1)
        .when(col('batch') == 'NG1_2024_7809', 4)
        .when(col('batch') == 'NG1_2024_7810', 4)
        .otherwise(col('reel_humidite_malt'))
    )
)

In [0]:
# Calcul du R2_sec
df_main = df_main.withColumn("reel_r2_sec",
(col("reel_malt_humide") - (col("reel_malt_humide") * (col("reel_humidite_malt") / 100))) /
(col("reel_orge_humide") - (col("reel_orge_humide") * (col("reel_humidite_orge") / 100)))* 100)

In [0]:
# Conversion en SQL
df_main.createOrReplaceTempView("main")

main = f"""
SELECT m.*, b.specification
FROM main m
LEFT JOIN {source_temp}.baseline b ON m.id_cahier_des_charges = b.id_specification AND m.id_site_production = id_site
"""

df_main = spark.sql(main)

# Remplacer les valeurs vides ou nulles par "monthly_average"
df_main = df_main.withColumn(
    "specification",
    when(col("specification").isNull() | (col("specification") == ""), "monthly_average").otherwise(col("specification")))

# Conversion en SQL
df_main.createOrReplaceTempView("main")

In [0]:
# Ajout des baseline par CDC, site de production et par mois
main_baseline = f"""
SELECT
  m.*,
  b.id_specification,
  b.nb_individus,
  b.Baseline_malt_yield_r2,
  b.Baseline_malt_dry_yield,
  b.Baseline_goods_weight,
  b.Baseline_goods_moisture,
  b.Baseline_malt_moisture,
  b.Baseline_FAN,
  b.Baseline_friabilite,
  b.Baseline_coloration_EBC,
  b.Baseline_betaG,
  b.Baseline_quality
FROM main m
LEFT JOIN {source_temp}.baseline b ON b.specification = m.specification AND b.id_site = m.id_site_production AND b.Month = MONTH(m.date_fin_production)
"""

df_main_baseline = spark.sql(main_baseline)

# Supprimer la colonne 'id_specification' et 'specification'
df_main_baseline = df_main_baseline.drop("id_specification", "specification")

In [0]:
# Renommer les colonnes baseline
df_main_baseline = df_main_baseline.withColumnRenamed("nb_individus", "nb_individus_baseline")
df_main_baseline = df_main_baseline.withColumnRenamed("Baseline_malt_yield_r2", "baseline_r2_humide")
df_main_baseline = df_main_baseline.withColumnRenamed("Baseline_malt_dry_yield", "baseline_r2_sec")
df_main_baseline = df_main_baseline.withColumnRenamed("Baseline_goods_weight", "baseline_orge_humide")
df_main_baseline = df_main_baseline.withColumnRenamed("Baseline_goods_moisture", "baseline_humidite_orge")
df_main_baseline = df_main_baseline.withColumnRenamed("Baseline_malt_moisture", "baseline_humidite_malt")
df_main_baseline = df_main_baseline.withColumnRenamed("Baseline_quality", "baseline_indice_qualite")

In [0]:
# Création d'une nouvelle colonne 'type_de_baseline' basée sur la condition de 'nb_individus_baseline'
df_main_baseline = df_main_baseline.withColumn(
    'type_de_baseline',
    F.when(
        F.col('nb_individus_baseline') >= 5,
        F.lit("Nombre d’individus suffisant")
    ).otherwise(F.lit("Baseline comblée"))
)

In [0]:
columns = ['baseline_humidite_orge', 'baseline_r2_humide', 'baseline_r2_sec', 'baseline_humidite_malt',
           'reel_r2_humide', 'reel_r2_sec', 'reel_humidite_malt', 'reel_humidite_orge']

# Appliquer la division sur chaque colonne spécifiée
for column in columns:
    df_main_baseline = df_main_baseline.withColumn(column, col(column) / 100)

In [0]:
df_main_baseline = df_main_baseline.withColumn("baseline_orge_sec",col("baseline_orge_humide") - (col("baseline_orge_humide") * col("baseline_humidite_orge")))
df_main_baseline = df_main_baseline.withColumn("baseline_malt_sec",col("baseline_orge_sec") * col("baseline_r2_sec"))
df_main_baseline = df_main_baseline.withColumn("baseline_eau_dans_orge",col("baseline_orge_humide") * col("baseline_humidite_orge"))
df_main_baseline = df_main_baseline.withColumn("baseline_eau_dans_malt",col("baseline_malt_sec") / ( 1 - col("baseline_humidite_malt")) - col("baseline_malt_sec"))
df_main_baseline = df_main_baseline.withColumn("baseline_malt_humide",col("baseline_eau_dans_malt") + col("baseline_malt_sec"))

df_main_baseline = df_main_baseline.withColumn("reel_eau_dans_orge",col("reel_orge_humide") * col("reel_humidite_orge"))
df_main_baseline = df_main_baseline.withColumn("reel_orge_sec",col("reel_orge_humide") - (col("reel_orge_humide") * col("reel_humidite_orge")))
df_main_baseline = df_main_baseline.withColumn("reel_malt_sec",col("reel_orge_sec") * col("reel_r2_sec"))
df_main_baseline = df_main_baseline.withColumn("reel_ecart_r2_sec_comparaison_baseline",col("reel_r2_sec") - col("baseline_r2_sec"))
df_main_baseline = df_main_baseline.withColumn("reel_ecart_orge_sec_comparaison_baseline",col("reel_orge_sec") - col("baseline_orge_sec"))

df_main_baseline = df_main_baseline.withColumn("malt_sec_supplementaire_via_perf_r2_sec",col("reel_ecart_r2_sec_comparaison_baseline") * col("baseline_orge_sec"))
df_main_baseline = df_main_baseline.withColumn("malt_sec_supplementaire_via_taille_batch_equivalent_baseline",col("reel_ecart_orge_sec_comparaison_baseline") * col("baseline_r2_sec"))
df_main_baseline = df_main_baseline.withColumn("malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec",
                             col("reel_ecart_r2_sec_comparaison_baseline") * col("reel_ecart_orge_sec_comparaison_baseline"))

df_main_baseline = df_main_baseline.withColumn("reel_eau_dans_malt",col("reel_malt_sec") / (1 - col("reel_humidite_malt")) - col("reel_malt_sec"))
df_main_baseline = df_main_baseline.withColumn("reel_malt_humide",col("reel_eau_dans_malt") + col("reel_malt_sec"))
df_main_baseline = df_main_baseline.withColumn("eau_equivalent_baseline",col("baseline_malt_sec") / (1 - col("baseline_humidite_malt")) - col("baseline_malt_sec"))

df_main_baseline = df_main_baseline.withColumn('eau_supplementaire_via_perf_humidite_malt',
                             col('baseline_malt_sec') / (1 - col('reel_humidite_malt')) - col('baseline_malt_sec') - col('eau_equivalent_baseline'))

df_main_baseline = df_main_baseline.withColumn('eau_supplementaire_via_perf_r2_sec_equivalent_baseline',
                             col('malt_sec_supplementaire_via_perf_r2_sec') / (1 - col('baseline_humidite_malt')) - col('malt_sec_supplementaire_via_perf_r2_sec'))

df_main_baseline = df_main_baseline.withColumn('eau_supplementaire_via_perf_r2_sec_et_perf_humidite_malt',
                             (col('malt_sec_supplementaire_via_perf_r2_sec') / (1 - col('reel_humidite_malt')) - col('malt_sec_supplementaire_via_perf_r2_sec')) -
                             col('eau_supplementaire_via_perf_r2_sec_equivalent_baseline'))

df_main_baseline = df_main_baseline.withColumn('eau_supplementaire_via_taille_batch_equivalent_baseline',
                             col('malt_sec_supplementaire_via_taille_batch_equivalent_baseline') / (1 - col('baseline_humidite_malt')) -
                             col('malt_sec_supplementaire_via_taille_batch_equivalent_baseline'))
                            
df_main_baseline = df_main_baseline.withColumn('eau_supplementaire_via_taille_batch_et_perf_humidite_malt',
                             (col('malt_sec_supplementaire_via_taille_batch_equivalent_baseline') / (1 - col('reel_humidite_malt')) -
                              col('malt_sec_supplementaire_via_taille_batch_equivalent_baseline')) - col('eau_supplementaire_via_taille_batch_equivalent_baseline'))

df_main_baseline = df_main_baseline.withColumn("eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline",
                             col("malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec") / (1 - col("baseline_humidite_malt")) -
                             col("malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec"))

df_main_baseline = df_main_baseline.withColumn("eau_supplementaire_via_taille_batch_et_perf_r2_sec_et_perf_humidite_malt",
                             (col("malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec") / (1 - col("reel_humidite_malt")) -
                              col("malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec")) - col("eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline"))

df_main_baseline = df_main_baseline.withColumn('R2_H_malt_sec',col('malt_sec_supplementaire_via_perf_r2_sec') + col('malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec'))

df_main_baseline = df_main_baseline.withColumn('R2_H_eau',col('eau_supplementaire_via_perf_humidite_malt') + col('eau_supplementaire_via_perf_r2_sec_equivalent_baseline') + 
                             col('eau_supplementaire_via_perf_r2_sec_et_perf_humidite_malt') + col('eau_supplementaire_via_taille_batch_et_perf_humidite_malt') +
                             col('eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline') + 
                             col('eau_supplementaire_via_taille_batch_et_perf_r2_sec_et_perf_humidite_malt'))

df_main_baseline = df_main_baseline.withColumn('total_malt_sec_supplementaire',
                             col('malt_sec_supplementaire_via_perf_r2_sec') + col('malt_sec_supplementaire_via_taille_batch_equivalent_baseline') +
                             col('malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec'))

df_main_baseline = df_main_baseline.withColumn('total_eau_supplementaire',
                             col('eau_supplementaire_via_perf_humidite_malt') + col('eau_supplementaire_via_perf_r2_sec_equivalent_baseline') +
                             col('eau_supplementaire_via_perf_r2_sec_et_perf_humidite_malt') + col('eau_supplementaire_via_taille_batch_equivalent_baseline') + 
                             col('eau_supplementaire_via_taille_batch_et_perf_humidite_malt') +
                             col('eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline') + 
                             col('eau_supplementaire_via_taille_batch_et_perf_r2_sec_et_perf_humidite_malt'))

df_main_baseline = df_main_baseline.withColumn('total_malt_humide_supplementaire',col('total_malt_sec_supplementaire') + col('total_eau_supplementaire'))

df_main_baseline = df_main_baseline.withColumn('gain_avec_IA',col('R2_H_malt_sec') + col('R2_H_eau') + col('eau_supplementaire_via_taille_batch_equivalent_baseline'))

# Créer une vue temporaire pour le DataFrame
df_main_baseline.createOrReplaceTempView("main_baseline")

In [0]:
bqt = f"""
SELECT b.fabrication_order_number, pe.code, btq.upper_limit, btq.lower_limit
FROM {source}.batches_quality_targets btq
JOIN {source}.batches b ON b.id_batch = btq.batch AND b.deleted = false
JOIN {source}.parameters_evaluations pe ON btq.evaluation_parameter = pe.id_parameter_evaluation
WHERE btq.deleted = false AND b.fabrication_order_number IS NOT NULL
"""

df_bqt = spark.sql(bqt)

In [0]:
# Max upper_limit
df_bqt_max = df_bqt.groupBy("fabrication_order_number").pivot("code").agg(F.max("upper_limit"))

# Renommer les colonnes dans df_bqt_max
df_bqt_max = df_bqt_max.withColumnRenamed('malt_color_ebc', 'coloration_EBC_max') \
                       .withColumnRenamed('malt_free_amino_nitrogen', 'FAN_max') \
                       .withColumnRenamed('malt_friability', 'friabilite_max') \
                       .withColumnRenamed('malt_soluble_betaglucan', 'betaG_max') \
                       .withColumnRenamed('product_moisture', 'hum_malt_max')

# Max lower_limit
df_bqt_min = df_bqt.groupBy("fabrication_order_number").pivot("code").agg(F.max("lower_limit"))

# Renommer les colonnes dans df_bqt_min
df_bqt_min = df_bqt_min.withColumnRenamed('malt_color_ebc', 'coloration_EBC_min') \
                       .withColumnRenamed('malt_free_amino_nitrogen', 'FAN_min') \
                       .withColumnRenamed('malt_friability', 'friabilite_min') \
                       .withColumnRenamed('malt_soluble_betaglucan', 'betaG_min') \
                       .withColumnRenamed('product_moisture', 'hum_malt_min')

# Jointure sur 'batch_number'
df_bqt = df_bqt_max.join(df_bqt_min.select(
    'fabrication_order_number', 
    'coloration_EBC_min', 
    'FAN_min', 
    'friabilite_min', 
    'betaG_min', 
    'hum_malt_min'
), on='fabrication_order_number', how='left')

# Sélection des colonnes finales
df_bqt = df_bqt.select(
    'fabrication_order_number',
    'FAN_max',
    'FAN_min',
    'hum_malt_max',
    'hum_malt_min',
    'betaG_max',
    'betaG_min',
    'friabilite_max',
    'friabilite_min',
    'coloration_EBC_max',
    'coloration_EBC_min'
)

# Créer une vue temporaire pour le DataFrame
df_bqt.createOrReplaceTempView("batches_quality_targets_min_max")

In [0]:
indice_qualite = f"""
SELECT
    fabrication_order,
    malt_free_amino_nitrogen AS FAN,
    malt_soluble_beta_glucan AS betaG,
    malt_friability AS friabilite,
    malt_moisture AS hum_malt,
    malt_color_ebc AS coloration_EBC,
    bqtmm.*
FROM {source_gold}.malt_quality_results mqr 
LEFT JOIN batches_quality_targets_min_max bqtmm ON bqtmm.fabrication_order_number = mqr.fabrication_order
"""

df_indice_qualite = spark.sql(indice_qualite)

In [0]:
# Calcul de FAN_result
df_indice_qualite = df_indice_qualite.withColumn(
    'FAN_result',
    F.when(
        (F.coalesce(F.col('FAN_min'), F.lit(float('-inf'))) <= F.col('FAN')) &
        (F.col('FAN') <= F.coalesce(F.col('FAN_max'), F.lit(float('inf')))),
        0.2
    ).otherwise(0)
)


# Calcul de hum_malt_result
df_indice_qualite = df_indice_qualite.withColumn(
    'hum_malt_result',
    F.when(
        (F.coalesce(F.col('hum_malt_min'), F.lit(float('-inf'))) <= F.col('hum_malt')) &
        (F.col('hum_malt') <= F.coalesce(F.col('hum_malt_max'), F.lit(float('inf')))),
        0.2
    ).otherwise(0)
)

# Calcul de betaG_result
df_indice_qualite = df_indice_qualite.withColumn(
    'betaG_result',
    F.when(
        (F.coalesce(F.col('betaG_min'), F.lit(float('-inf'))) <= F.col('betaG')) &
        (F.col('betaG') <= F.coalesce(F.col('betaG_max'), F.lit(float('inf')))),
        0.2
    ).otherwise(0)
)

# Calcul de friabilite_result
df_indice_qualite = df_indice_qualite.withColumn(
    'friabilite_result',
    F.when(
        (F.coalesce(F.col('friabilite_min'), F.lit(float('-inf'))) <= F.col('friabilite')) &
        (F.col('friabilite') <= F.coalesce(F.col('friabilite_max'), F.lit(float('inf')))),
        0.2
    ).otherwise(0)
)


# Calcul de coloration_EBC_result
df_indice_qualite = df_indice_qualite.withColumn(
    'coloration_EBC_result',
    F.when(
        (F.coalesce(F.col('coloration_EBC_min'), F.lit(float('-inf'))) <= F.col('coloration_EBC')) &
        (F.col('coloration_EBC') <= F.coalesce(F.col('coloration_EBC_max'), F.lit(float('inf')))),
        0.2
    ).otherwise(0)
)

# Calcul de l'indice_qualite
df_indice_qualite = df_indice_qualite.withColumn(
    'indice_qualite', 
    F.round(
        F.col('FAN_result') + F.col('hum_malt_result') + F.col('betaG_result') + F.col('friabilite_result') + F.col('coloration_EBC_result'),1))

In [0]:
# Liste des colonnes à reformater en float
colonnes_a_reformater = [
    'FAN_min', 'FAN_max', 'FAN_result',
    'hum_malt_min', 'hum_malt_max', 'hum_malt_result',
    'betaG_min', 'betaG_max', 'betaG_result',
    'friabilite_min', 'friabilite_max', 'friabilite_result',
    'coloration_EBC_min', 'coloration_EBC_max', 'coloration_EBC_result'
]

# Reformate chaque colonne en float
for colonne in colonnes_a_reformater:
    df_indice_qualite = df_indice_qualite.withColumn(colonne, F.col(colonne).cast("float"))

In [0]:
df_indice_qualite = df_indice_qualite.select(
    'fabrication_order',
    'FAN', 'FAN_max', 'FAN_min', 'FAN_result',
    'hum_malt', 'hum_malt_max', 'hum_malt_min', 'hum_malt_result',
    'betaG', 'betaG_max', 'betaG_min', 'betaG_result',
    'friabilite', 'friabilite_max', 'friabilite_min', 'friabilite_result',
    'coloration_EBC', 'coloration_EBC_max', 'coloration_EBC_min', 'coloration_EBC_result',
    'indice_qualite'
)

# Créer une vue temporaire pour le DataFrame
df_indice_qualite.createOrReplaceTempView("indice_qualite")

In [0]:
fact = f"""
SELECT mb.*,
        iq.FAN, iq.FAN_max, iq.FAN_min, iq.FAN_result,
        iq.hum_malt, iq.hum_malt_max, iq.hum_malt_min, iq.hum_malt_result,
        iq.betaG, iq.betaG_max, iq.betaG_min, iq.betaG_result,
        iq.friabilite, iq.friabilite_max, iq.friabilite_min, iq.friabilite_result,
        iq.coloration_EBC, iq.coloration_EBC_max, iq.coloration_EBC_min, iq.coloration_EBC_result,
        iq.indice_qualite,
        a.reco_acceptee, t.reco_trempe, g.reco_germination, to.reco_touraille
FROM main_baseline mb

LEFT JOIN indice_qualite iq ON iq.fabrication_order = mb.ordre_de_fabrication

LEFT JOIN (SELECT r.batch, SUM(r.acceptance_status) AS reco_acceptee
        FROM {source}.recommendations r
        WHERE r.acceptance_status = 1
        GROUP BY r.batch) a ON a.batch = mb.id_batch

LEFT JOIN (SELECT r.batch, SUM(r.acceptance_status) AS reco_trempe
        FROM {source}.recommendations r
        JOIN {source}.parameters_localizations_translations plt ON plt.id_parameter_localization = r.target_localization
        WHERE plt.language = 1 AND plt.label LIKE 'Trempe%' AND r.acceptance_status = 1
        GROUP BY r.batch) t ON t.batch = mb.id_batch

LEFT JOIN (SELECT r.batch, SUM(r.acceptance_status) AS reco_germination
        FROM {source}.recommendations r
        JOIN {source}.parameters_localizations_translations plt ON plt.id_parameter_localization = r.target_localization
        WHERE plt.language = 1 AND plt.label LIKE 'Germination%' AND r.acceptance_status = 1
        GROUP BY r.batch) g ON g.batch = mb.id_batch

LEFT JOIN (SELECT r.batch, SUM(r.acceptance_status) AS reco_touraille
        FROM {source}.recommendations r
        JOIN {source}.parameters_localizations_translations plt ON plt.id_parameter_localization = r.target_localization
        WHERE plt.language = 1 AND plt.label LIKE 'Touraille%' AND r.acceptance_status = 1
        GROUP BY r.batch) to ON to.batch = mb.id_batch
"""

df_fact = spark.sql(fact)

In [0]:
# Supprime les doublons
df_fact = df_fact.dropDuplicates()

In [0]:
# Remplacer les nulls par 0 dans les colonnes spécifiées
columns_to_update = ['reco_acceptee', 'reco_trempe', 'reco_germination', 'reco_touraille']
df_fact = df_fact.fillna(0, subset=columns_to_update)

In [0]:
# Renommer les colonnes dans df_bqt_min
df_fact = df_fact.withColumnRenamed('Baseline_FAN', 'baseline_FAN') \
                 .withColumnRenamed('Baseline_betaG', 'baseline_betaG') \
                 .withColumnRenamed('Baseline_coloration_EBC', 'baseline_coloration_EBC') \
                 .withColumnRenamed('Baseline_friabilite', 'baseline_friabilite')

In [0]:
df_fact = df_fact.select(
'id_batch',
'batch',
'ordre_de_fabrication',
'lot_mes',
'annee_de_recolte',
'id_site_production',
'site_production',
'date_fin_production',
'type_de_production',
'id_cahier_des_charges',
'cahier_des_charges',
'goods_specy',
'goods_variety',
'nb_individus_baseline',
'type_de_baseline',
'baseline_r2_sec',
'baseline_r2_humide',
'baseline_orge_humide',
'baseline_humidite_orge',
'baseline_orge_sec',
'baseline_eau_dans_orge',
'baseline_malt_humide',
'baseline_humidite_malt',
'baseline_malt_sec',
'baseline_eau_dans_malt',
'reel_r2_sec',
'reel_r2_humide',
'reel_orge_humide',
'reel_humidite_orge',
'reel_orge_sec',
'reel_eau_dans_orge',
'reel_malt_humide',
'reel_humidite_malt',
'reel_malt_sec',
'reel_eau_dans_malt',
'reel_ecart_r2_sec_comparaison_baseline',
'reel_ecart_orge_sec_comparaison_baseline',
'malt_sec_supplementaire_via_perf_r2_sec',
'malt_sec_supplementaire_via_taille_batch_equivalent_baseline',
'malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec',
'eau_equivalent_baseline',
'eau_supplementaire_via_perf_humidite_malt',
'eau_supplementaire_via_perf_r2_sec_equivalent_baseline',
'eau_supplementaire_via_perf_r2_sec_et_perf_humidite_malt',
'eau_supplementaire_via_taille_batch_equivalent_baseline',
'eau_supplementaire_via_taille_batch_et_perf_humidite_malt',
'eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline',
'eau_supplementaire_via_taille_batch_et_perf_r2_sec_et_perf_humidite_malt',
'total_malt_sec_supplementaire',
'total_eau_supplementaire',
'total_malt_humide_supplementaire',
'R2_H_malt_sec',
'R2_H_eau',
'gain_avec_IA',
'reco_acceptee',
'reco_trempe',
'reco_germination',
'reco_touraille',
'FAN',
'FAN_max',
'FAN_min',
'FAN_result',
'baseline_FAN',
'betaG',
'betaG_max',
'betaG_min',
'betaG_result',
'baseline_betaG',
'friabilite',
'friabilite_max',
'friabilite_min',
'friabilite_result',
'baseline_friabilite',
'coloration_EBC',
'coloration_EBC_max',
'coloration_EBC_min',
'coloration_EBC_result',
'baseline_coloration_EBC',
'hum_malt',
'hum_malt_max',
'hum_malt_min',
'hum_malt_result',
'indice_qualite',
'baseline_indice_qualite'
)

In [0]:
df_tableau = df_fact

In [0]:
df_erreur_fact = df_fact

# Identification des erreurs et manques de données dans les colonnes spécifiées
colonnes_erreur = ['id_site_production', 'site_production', 'date_fin_production', 'type_de_production', 
                   'id_cahier_des_charges', 'cahier_des_charges', 'baseline_orge_humide', 'baseline_r2_humide', 'baseline_humidite_orge', 
                   'baseline_r2_sec', 'baseline_humidite_malt', 'reel_orge_humide', 'reel_humidite_orge', 
                   'reel_humidite_malt', 'reel_malt_humide']

# Filtrer les lignes avec des données manquantes dans les colonnes spécifiées
condition = reduce(lambda x, y: x | y, [F.col(col).isNull() | (F.col(col) == '') for col in colonnes_erreur])
df_erreur_fact = df_erreur_fact.filter(condition)

# Création de la colonne 'causes' en listant les colonnes vides pour chaque ligne
df_erreur_fact = df_erreur_fact.withColumn(
    'causes',
    F.array([F.when(F.col(col).isNull() | (F.col(col) == ''), F.lit(f"{col} est vide")).otherwise(F.lit(None)) for col in colonnes_erreur])
)

# Supprimer les éléments vides de la liste 'causes'
df_erreur_fact = df_erreur_fact.withColumn('causes', F.expr("filter(causes, x -> x is not null)"))

# Explosion de la colonne 'causes'
df_erreur_fact = df_erreur_fact.withColumn('causes', F.explode('causes'))

# Filtrer les lignes où 'causes' commence par "Baseline" et 'date_fin_production' est vide
df_erreur_fact = df_erreur_fact.filter(~((F.col('causes').startswith('baseline')) & F.col('date_fin_production').isNull()))

# Sélectionner les colonnes finales
colonnes_finales = ['id_batch', 'batch', 'id_site_production', 'site_production', 'date_fin_production', 'type_de_production', 
                    'id_cahier_des_charges', 'cahier_des_charges', 'baseline_orge_humide', 'baseline_r2_humide', 'baseline_humidite_orge', 
                    'baseline_r2_sec', 'baseline_humidite_malt', 'reel_orge_humide', 'reel_humidite_orge', 
                    'reel_humidite_malt', 'reel_malt_humide', 'causes']
df_erreur_fact = df_erreur_fact.select(colonnes_finales)

# Supprimer les doublons dans le DataFrame
df_erreur_fact = df_erreur_fact.dropDuplicates()

In [0]:
#df_fact = df_fact.dropna(subset=colonnes_erreur)

Import table de faits

In [0]:
current_process="fact"

In [0]:
target_fact = current_catalog +"."+current_schema+"."+current_process
print(target_fact)

In [0]:
all_columns =  df_fact.columns
display(all_columns)

In [0]:
# define the primary key 
primary_key = [    
    'id_batch']

additional_columns = get_additional_columns(all_columns, primary_key)

In [0]:
if verbose_mode == 'debug': 
    print(additional_columns)


handle_table_update(
    df_fact, 
    target_fact, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

erreur_fact

In [0]:
# Changer les types de données
df_erreur_fact = df_erreur_fact.select(
    F.col('id_batch').cast(StringType()),
    F.col('batch').cast(StringType()),
    F.col('id_site_production').cast(IntegerType()),
    F.col('site_production').cast(StringType()),
    F.col('date_fin_production').cast(TimestampType()),
    F.col('type_de_production').cast(StringType()),
    F.col('id_cahier_des_charges').cast(IntegerType()),
    F.col('cahier_des_charges').cast(StringType()),
    F.col('baseline_orge_humide').cast(FloatType()),
    F.col('baseline_r2_humide').cast(FloatType()),
    F.col('baseline_humidite_orge').cast(FloatType()),
    F.col('baseline_r2_sec').cast(FloatType()),
    F.col('baseline_humidite_malt').cast(FloatType()),
    F.col('reel_orge_humide').cast(FloatType()),
    F.col('reel_humidite_orge').cast(FloatType()),
    F.col('reel_humidite_malt').cast(FloatType()),
    F.col('reel_malt_humide').cast(FloatType()),
    F.col('causes').cast(StringType())
)

Import erreur_fact

In [0]:
current_process="erreur_fact"

In [0]:
target_fact_erreur = current_catalog +"."+current_schema+"."+current_process
print(target_fact_erreur)

In [0]:
all_columns =  df_erreur_fact.columns
display(all_columns)

In [0]:
# define the primary key 
primary_key = [    
    'id_batch',
    'causes']

additional_columns = get_additional_columns(all_columns, primary_key)

In [0]:
if verbose_mode == 'debug': 
    print(additional_columns)


handle_table_update(
    df_erreur_fact, 
    target_fact_erreur, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full" # Use "update" for update mode, "full" for delete/insert mode
    )

Energies

In [0]:
# Créer une vue temporaire pour le DataFrame
df_fact.createOrReplaceTempView("fact")

In [0]:
nrj = f"""
SELECT
    f.batch, f.date_fin_production, f.id_cahier_des_charges, f.cahier_des_charges, f.type_de_production, f.id_site_production, f.site_production, f.goods_specy, f.goods_variety,
    cet.global_electric_energy_lhv_consumption, cet.global_thermal_energy_lhv_consumption, b.Baseline_electricity, b.Baseline_thermal,
    f.reel_r2_sec, f.reel_orge_humide, f.reel_humidite_orge, f.reel_humidite_malt
FROM {source_temp}.conso_energie_temp cet
JOIN fact f ON f.batch = cet.batch_number
JOIN {source_temp}.baseline b ON b.Specification = f.cahier_des_charges AND b.id_site = f.id_site_production AND b.Month = MONTH(f.date_fin_production)
"""

df_nrj = spark.sql(nrj)

In [0]:
# Dictionnaire des colonnes à renommer
colonnes = {
    'global_electric_energy_lhv_consumption': 'reel_elec_kWh_t',
    'global_thermal_energy_lhv_consumption': 'reel_thermique_kWh_t'
}

# Conversion des colonnes en valeurs numériques et traitement des NaN et des valeurs inférieures à 10
for colonne_originale, nouveau_nom in colonnes.items():
    df_nrj = df_nrj.withColumn(
        colonne_originale, 
        F.when(F.col(colonne_originale).cast('float') < 10, 0)
        .otherwise(F.col(colonne_originale).cast('float'))
    )
    # Remplacement des NaN par 0
    df_nrj = df_nrj.fillna({colonne_originale: 0})
    # Renommage des colonnes
    df_nrj = df_nrj.withColumnRenamed(colonne_originale, nouveau_nom)

columns = ['reel_r2_sec', 'reel_humidite_orge', 'reel_humidite_malt']

# Calcul de 'goods_dry_weight'
df_nrj = df_nrj.withColumn(
    'reel_orge_sec', 
    F.col('reel_orge_humide') - (F.col('reel_orge_humide') * F.col('reel_humidite_orge'))
)

# Calcul de 'malt_dry_weight'
df_nrj = df_nrj.withColumn(
    'reel_malt_sec', 
    F.col('reel_orge_humide') * F.col('reel_r2_sec')
)

# Calcul de 'reel_eau_dans_malt'
df_nrj = df_nrj.withColumn(
    'reel_eau_dans_malt', 
    (F.col('reel_malt_sec') / (1 - F.col('reel_humidite_malt'))) - F.col('reel_malt_sec')
)

# Calcul de 'malt_weight'
df_nrj = df_nrj.withColumn(
    'reel_malt_humide', 
    F.col('reel_eau_dans_malt') + F.col('reel_malt_sec')
)

In [0]:
colonnes = {
    'Baseline_electricity': 'baseline_elec_kWh_t',
    'Baseline_thermal': 'baseline_thermique_kWh_t'
}

# Renommer les colonnes
for colonne_originale, nouveau_nom in colonnes.items():
    df_nrj = df_nrj.withColumnRenamed(colonne_originale, nouveau_nom)

# Calcul des écarts
df_nrj = df_nrj.withColumn('ecart_thermique_kWh_t', col('reel_thermique_kWh_t') - col('baseline_thermique_kWh_t'))
df_nrj = df_nrj.withColumn('ecart_elec_kWh_t', col('reel_elec_kWh_t') - col('baseline_elec_kWh_t'))

# Appliquer la condition where
df_nrj = df_nrj.withColumn('ecart_elec_kWh_t', 
                           when(col('reel_elec_kWh_t') <= 0, 0).otherwise(col('ecart_elec_kWh_t')))

In [0]:
# Sélection des colonnes spécifiques
df_nrj = df_nrj.select(
    'batch',
    'date_fin_production',
    'id_cahier_des_charges',
    'cahier_des_charges',
    'type_de_production',
    'id_site_production',
    'site_production',
    'goods_specy',
    'goods_variety',
    'reel_malt_humide',
    'reel_thermique_kWh_t',
    'baseline_thermique_kWh_t',
    'ecart_thermique_kWh_t',
    'reel_elec_kWh_t',
    'baseline_elec_kWh_t',
    'ecart_elec_kWh_t'
)

In [0]:
df_erreur_nrj = df_nrj

# Identification des erreurs et manques de données dans les colonnes spécifiées
colonnes_erreur_nrj = ['reel_thermique_kWh_t', 'baseline_thermique_kWh_t', 'reel_elec_kWh_t', 'baseline_elec_kWh_t']

# Construire la condition pour vérifier les valeurs manquantes
condition = None
for col in colonnes_erreur_nrj:
    if condition is None:
        condition = F.col(col).isNull() | (F.col(col) == '')
    else:
        condition = condition | (F.col(col).isNull() | (F.col(col) == ''))

# Filtrer les lignes avec des données manquantes dans les colonnes spécifiées
df_erreur_nrj = df_erreur_nrj.filter(condition)

# Création de la colonne 'causes' en listant les colonnes vides pour chaque ligne
df_erreur_nrj = df_erreur_nrj.withColumn(
    'causes',
    F.array([F.when(F.col(col).isNull() | (F.col(col) == ''), F.lit(f"{col} est vide")).otherwise(F.lit(None)) for col in colonnes_erreur_nrj])
)

# Supprimer les éléments nuls de la liste 'causes'
df_erreur_nrj = df_erreur_nrj.withColumn('causes', F.expr("filter(causes, x -> x is not null)"))

# Explosion de la colonne 'causes'
df_erreur_nrj = df_erreur_nrj.withColumn('causes', F.explode('causes'))

# Filtrer les lignes où 'causes' commence par "Baseline" et 'date_fin_production' est vide
df_erreur_nrj = df_erreur_nrj.filter(~((F.col('causes').startswith('baseline')) & F.col('date_fin_production').isNull()))

# Sélectionner les colonnes finales
colonnes_finales = ['batch', 'id_site_production', 'site_production', 'date_fin_production', 'type_de_production', 
                    'id_cahier_des_charges', 'cahier_des_charges', 'reel_thermique_kWh_t', 'baseline_thermique_kWh_t', 'reel_elec_kWh_t', 
                    'baseline_elec_kWh_t', 'causes']
df_erreur_nrj = df_erreur_nrj.select(colonnes_finales)

In [0]:
df_nrj = df_nrj.dropDuplicates(["batch"])

In [0]:
# Supprimer les lignes où une ou plusieurs colonnes dans 'colonnes' ont des valeurs manquantes
df_nrj = df_nrj.dropna(subset=colonnes_erreur_nrj)

In [0]:
# Changer les types de données
df_nrj = df_nrj.select(
    F.col('batch').cast(StringType()),
    F.col('date_fin_production').cast(TimestampType()),
    F.col('id_cahier_des_charges').cast(IntegerType()),
    F.col('cahier_des_charges').cast(StringType()),
    F.col('type_de_production').cast(StringType()),
    F.col('id_site_production').cast(IntegerType()),
    F.col('site_production').cast(StringType()),
    F.col('goods_specy').cast(StringType()),
    F.col('goods_variety').cast(StringType()),
    F.col('reel_malt_humide').cast(FloatType()),
    F.col('reel_thermique_kWh_t').cast(FloatType()),
    F.col('baseline_thermique_kWh_t').cast(FloatType()),
    F.col('ecart_thermique_kWh_t').cast(FloatType()),
    F.col('reel_elec_kWh_t').cast(FloatType()),
    F.col('baseline_elec_kWh_t').cast(FloatType()),
    F.col('ecart_elec_kWh_t').cast(FloatType()),
)

In [0]:
# Ajouter les colonnes 'created_at', 'updated_at' et 'deleted_at' dans le dataframe
df_nrj = df_nrj \
    .withColumn('created_at', lit(None).cast('timestamp')) \
    .withColumn('updated_at', lit(None).cast('timestamp')) \
    .withColumn('deleted_at', lit(None).cast('timestamp'))

Import fact_kWh_t

In [0]:
current_process="fact_kWh_t"

In [0]:
target_fact_kWh_t = current_catalog +"."+current_schema+"."+current_process
print(target_fact_kWh_t)

In [0]:
all_columns =  df_nrj.columns
display(all_columns)

In [0]:
# define the primary key 
primary_key = ['batch']

additional_columns = get_additional_columns(all_columns, primary_key)

In [0]:
if verbose_mode == 'debug': 
    print(additional_columns)


handle_table_update(
    df_nrj, 
    target_fact_kWh_t, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

MWh

In [0]:
from pyspark.sql.functions import col

# Création d'un dataframe df_MWh qui est une "copie" de df_nrj
df_MWh = df_nrj

# Calcul des valeurs en MWh pour les colonnes thermique et électrique
df_MWh = df_MWh.withColumn('reel_thermique_MWh', (col('reel_thermique_kWh_t') * col('reel_malt_humide')) / 1000)
df_MWh = df_MWh.withColumn('reel_elec_MWh', (col('reel_elec_kWh_t') * col('reel_malt_humide')) / 1000)
df_MWh = df_MWh.withColumn('baseline_thermique_MWh', (col('baseline_thermique_kWh_t') * col('reel_malt_humide')) / 1000)
df_MWh = df_MWh.withColumn('baseline_elec_MWh', (col('baseline_elec_kWh_t') * col('reel_malt_humide')) / 1000)

# Calcul des écarts en MWh
df_MWh = df_MWh.withColumn('ecart_thermique_MWh', col('reel_thermique_MWh') - col('baseline_thermique_MWh'))
df_MWh = df_MWh.withColumn('ecart_elec_MWh', col('reel_elec_MWh') - col('baseline_elec_MWh'))

# Sélection des colonnes spécifiques
df_MWh = df_MWh.select(
    'batch',
    'date_fin_production',
    'id_cahier_des_charges',
    'cahier_des_charges',
    'type_de_production',
    'id_site_production',
    'site_production',
    'goods_specy',
    'goods_variety',
    'reel_malt_humide',
    'reel_thermique_MWh',
    'baseline_thermique_MWh',
    'ecart_thermique_MWh',
    'reel_elec_MWh',
    'baseline_elec_MWh',
    'ecart_elec_MWh'
)

Import fact_MWh

In [0]:
current_process="fact_MWh"

In [0]:
target_fact_MWh = current_catalog +"."+current_schema+"."+current_process
print(target_fact_MWh)

In [0]:
all_columns =  df_MWh.columns
display(all_columns)

In [0]:
# define the primary key 
primary_key = [    
    'batch']

additional_columns = get_additional_columns(all_columns, primary_key)

In [0]:
if verbose_mode == 'debug': 
    print(additional_columns)


handle_table_update(
    df_MWh, 
    target_fact_MWh, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

erreur_nrj

In [0]:
# Changer les types de données
df_erreur_nrj = df_erreur_nrj.select(
    F.col('batch').cast(StringType()),
    F.col('id_site_production').cast(IntegerType()),
    F.col('site_production').cast(StringType()),
    F.col('date_fin_production').cast(TimestampType()),
    F.col('type_de_production').cast(StringType()),
    F.col('id_cahier_des_charges').cast(IntegerType()),
    F.col('cahier_des_charges').cast(StringType()),
    F.col('reel_thermique_kWh_t').cast(FloatType()),
    F.col('baseline_thermique_kWh_t').cast(FloatType()),
    F.col('reel_elec_kWh_t').cast(FloatType()),
    F.col('baseline_elec_kWh_t').cast(FloatType()),
    F.col('causes').cast(StringType())
)

Import erreur_nrj

In [0]:
current_process="erreur_nrj"

In [0]:
target_fact_erreur_nrj = current_catalog +"."+current_schema+"."+current_process
print(target_fact_erreur_nrj)

In [0]:
all_columns =  df_erreur_nrj.columns
display(all_columns)

In [0]:
# define the primary key 
primary_key = [    
    'batch']

additional_columns = get_additional_columns(all_columns, primary_key)

In [0]:
if verbose_mode == 'debug': 
    print(additional_columns)


handle_table_update(
    df_erreur_nrj, 
    target_fact_erreur_nrj, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

fact_review

In [0]:
df_tableau = df_tableau.select(
    'id_batch',
    'date_fin_production',
    'malt_sec_supplementaire_via_perf_r2_sec',
    'malt_sec_supplementaire_via_taille_batch_equivalent_baseline',
    'malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec',
    'eau_supplementaire_via_perf_humidite_malt',
    'eau_supplementaire_via_perf_r2_sec_equivalent_baseline',
    'eau_supplementaire_via_perf_r2_sec_et_perf_humidite_malt',
    'eau_supplementaire_via_taille_batch_equivalent_baseline',
    'eau_supplementaire_via_taille_batch_et_perf_humidite_malt',
    'eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline',
    'eau_supplementaire_via_taille_batch_et_perf_r2_sec_et_perf_humidite_malt',
    'R2_H_malt_sec',
    'R2_H_eau',
    'reel_malt_humide',
    'baseline_malt_humide'
)

In [0]:
# Ajouter une colonne fixe
df_tableau = df_tableau.withColumn('nb_de_couche', F.lit(1))

# Ajouter une colonne calculée
df_tableau = df_tableau.withColumn(
    'malt_humide_supp_baseline',
    F.col('malt_sec_supplementaire_via_taille_batch_equivalent_baseline') +
    F.col('eau_supplementaire_via_taille_batch_equivalent_baseline') +
    F.col('R2_H_malt_sec') +
    F.col('R2_H_eau')
)

In [0]:
# Conversion de la colonne 'date_fin_production' en type datetime
df_tableau = df_tableau.withColumn('date_fin_production', F.to_date(F.col('date_fin_production')))

# Ajout de la colonne 'mois_annee' avec le format "janvier 2024"
df_tableau = df_tableau.withColumn(
    'mois_annee',
    F.concat(
        F.when(F.month('date_fin_production') == 1, F.lit('janvier'))
         .when(F.month('date_fin_production') == 2, F.lit('février'))
         .when(F.month('date_fin_production') == 3, F.lit('mars'))
         .when(F.month('date_fin_production') == 4, F.lit('avril'))
         .when(F.month('date_fin_production') == 5, F.lit('mai'))
         .when(F.month('date_fin_production') == 6, F.lit('juin'))
         .when(F.month('date_fin_production') == 7, F.lit('juillet'))
         .when(F.month('date_fin_production') == 8, F.lit('août'))
         .when(F.month('date_fin_production') == 9, F.lit('septembre'))
         .when(F.month('date_fin_production') == 10, F.lit('octobre'))
         .when(F.month('date_fin_production') == 11, F.lit('novembre'))
         .otherwise(F.lit('décembre')),
        F.lit(" "),
        F.year('date_fin_production').cast("string")
    )
)

df_tableau = df_tableau.withColumn("date_fin_production", to_date("date_fin_production", "dd/MM/yyyy")).withColumn("custom_order", date_format("date_fin_production", "yyyyMM"))

df_tableau = df_tableau.drop('date_fin_production')

In [0]:
# Liste des colonnes à pivoter
colonnes_a_pivoter = [col for col in df_tableau.columns if col not in ['id_batch', 'mois_annee', 'custom_order']]

# Création d'une structure clé-valeur pour les colonnes à pivoter
df_tableau = df_tableau.withColumn(
    'ventilation_valeurs',
    F.explode(
        F.array(*[
            F.struct(F.lit(col).alias('ventilation'), F.col(col).alias('valeur'))
            for col in colonnes_a_pivoter
        ])
    )
)

# Extraction des colonnes "ventilation" et "valeur" depuis la structure
df_tableau = df_tableau.select(
    'id_batch',
    'mois_annee',
    'custom_order',
    F.col('ventilation_valeurs.ventilation').alias('ventilation'),
    F.col('ventilation_valeurs.valeur').alias('valeur')
)

In [0]:
# Ajout de la colonne 'id_ventilation'
df_tableau = df_tableau.withColumn(
    'id_ventilation',
    F.when(F.col('ventilation') == 'malt_sec_supplementaire_via_perf_r2_sec', 1)
     .when(F.col('ventilation') == 'malt_sec_supplementaire_via_taille_batch_equivalent_baseline', 2)
     .when(F.col('ventilation') == 'malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec', 3)
     .when(F.col('ventilation') == 'eau_supplementaire_via_perf_humidite_malt', 4)
     .when(F.col('ventilation') == 'eau_supplementaire_via_perf_r2_sec_equivalent_baseline', 5)
     .when(F.col('ventilation') == 'eau_supplementaire_via_perf_r2_sec_et_perf_humidite_malt', 6)
     .when(F.col('ventilation') == 'eau_supplementaire_via_taille_batch_equivalent_baseline', 7)
     .when(F.col('ventilation') == 'eau_supplementaire_via_taille_batch_et_perf_humidite_malt', 8)
     .when(F.col('ventilation') == 'eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline', 9)
     .when(F.col('ventilation') == 'eau_supplementaire_via_taille_batch_et_perf_r2_sec_et_perf_humidite_malt', 10)
     .when(F.col('ventilation') == 'nb_de_couche', 0)
     .when(F.col('ventilation') == 'baseline_malt_humide', 12)
     .when(F.col('ventilation') == 'reel_malt_humide', 13)
     .when(F.col('ventilation') == 'R2_H_malt_sec', 14)
     .when(F.col('ventilation') == 'R2_H_eau', 15)
     .when(F.col('ventilation') == 'malt_humide_supp_baseline', 16)
)

In [0]:
df_dim_tableau = df_tableau

In [0]:
df_tableau = df_tableau.select('id_batch','id_ventilation','valeur', 'mois_annee','custom_order')

In [0]:
df_tableau = df_tableau.withColumn(
    "valeur",
    F.col("valeur").cast("float")
)

df_tableau = df_tableau.withColumn(
    "custom_order",
    F.col("custom_order").cast("int")
)

Import fact_review

In [0]:
current_process="fact_review"

In [0]:
target_fact_review = current_catalog +"."+current_schema+"."+current_process
print(target_fact_review)

In [0]:
all_columns =  df_tableau.columns
display(all_columns)

In [0]:
# define the primary key 
primary_key = [    
    'id_batch',
    'id_ventilation']

additional_columns = get_additional_columns(all_columns, primary_key)

In [0]:
if verbose_mode == 'debug': 
    print(additional_columns)


handle_table_update(
    df_tableau, 
    target_fact_review, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full" # Use "update" for update mode, "full" for delete/insert mode
    )

dim_review

In [0]:
df_dim_tableau = df_dim_tableau.drop('batch', 'mois', 'valeur')

In [0]:
# Calcul de la moyenne par 'ventilation'
df_dim_tableau = df_dim_tableau.groupBy('ventilation').mean()
for column in df_dim_tableau.columns:  # Remplace 'col' par 'column'
    if column.startswith("avg("):
        df_dim_tableau = df_dim_tableau.withColumnRenamed(column, column[4:-1])

In [0]:
# Ajouter la colonne 'texte' basée sur le mappage de 'ventilation'
df_dim_tableau = df_dim_tableau.withColumn(
    'texte',
    F.when(F.col('ventilation') == 'malt_sec_supplementaire_via_perf_r2_sec', 'Tonne Malt sec supplémentaire lié à un meilleur R2 sec')
     .when(F.col('ventilation') == 'malt_sec_supplementaire_via_taille_batch_equivalent_baseline', "Tonne Malt sec supplémentaire lié à l'augmentation du batch")
     .when(F.col('ventilation') == 'malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec', "Tonne Malt sec supplémentaire lié à un meilleur R2 sec (part issue de l'augmentation du batch)")
     .when(F.col('ventilation') == 'eau_supplementaire_via_perf_humidite_malt', 'Eau supplémentaire liée à une meilleure humidité')
     .when(F.col('ventilation') == 'eau_supplementaire_via_perf_r2_sec_equivalent_baseline', 'Eau supplémentaire liée à un meilleur R2 sec')
     .when(F.col('ventilation') == 'eau_supplementaire_via_perf_r2_sec_et_perf_humidite_malt', 'Eau supplémentaire lié à un meilleur R2 sec et une meilleur humidité')
     .when(F.col('ventilation') == 'eau_supplementaire_via_perf_et_humidite_malt', 'Eau supplémentaire liée à un meilleur R2 sec et une meilleure humidité')
     .when(F.col('ventilation') == 'eau_supplementaire_via_taille_batch_equivalent_baseline', "Eau supplémentaire liée à l'augmentation du batch")
     .when(F.col('ventilation') == 'eau_supplementaire_via_taille_batch_et_perf_humidite_malt', "Eau supplémentaire liée à une meilleure humidité (part issue de l'augmentation du batch)")
     .when(F.col('ventilation') == 'eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline', "Eau supplémentaire liée à un meilleur R2 sec (part issue de l'augmentation du batch)")
     .when(F.col('ventilation') == 'eau_supplementaire_via_taille_batch_et_perf_r2_sec_et_perf_humidite_malt', "Eau supplémentaire liée à une meilleure humidité (part issue de l'augmentation du batch et d'un meilleur R2)")
     .when(F.col('ventilation') == 'nb_de_couche', 'Nombre de couche')
     .when(F.col('ventilation') == 'baseline_malt_humide', 'Total Tonnes Malt Humide Equivalent Baseline')
     .when(F.col('ventilation') == 'reel_malt_humide', 'Total Tonnes Malt Humide réel')
     .when(F.col('ventilation') == 'R2_H_malt_sec', 'Tonnes Malt sec supplémentaires liées à la performance du R2sec/Gain en eau')
     .when(F.col('ventilation') == 'R2_H_eau', 'Tonnes Eau supplémentaires liées à la performance du R2sec/Gain en eau')
     .when(F.col('ventilation') == 'malt_humide_supp_baseline', 'Total Tonnes Malt Humide supplémentaires produites par le site rapport à la Baseline')
     .otherwise(None)  # Valeur par défaut si aucune correspondance
)

In [0]:
df_dim_tableau = df_dim_tableau.select('id_ventilation', 'ventilation', 'texte')

Import dim_review

In [0]:
current_process="dim_review"

In [0]:
target_dim_review = current_catalog +"."+current_schema+"."+current_process
print(target_dim_review)

In [0]:
all_columns =  df_dim_tableau.columns
display(all_columns)

In [0]:
# define the primary key 
primary_key = [    
    'id_ventilation']

additional_columns = get_additional_columns(all_columns, primary_key)

In [0]:
if verbose_mode == 'debug': 
    print(additional_columns)


handle_table_update(
    df_dim_tableau, 
    target_dim_review, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

dim_specification

In [0]:
dim_cdc = f"""
SELECT id_requirement_specification AS id_cdc, name AS cdc
FROM {source}.requirement_specifications
WHERE deleted = false
"""

df_dim_cdc = spark.sql(dim_cdc)

Import dim_specification

In [0]:
current_process="dim_specification"

In [0]:
target_dim_specification = current_catalog +"."+current_schema+"."+current_process
print(target_dim_specification)

In [0]:
all_columns =  df_dim_cdc.columns
display(all_columns)

In [0]:
# define the primary key 
primary_key = [    
    'id_cdc']

additional_columns = get_additional_columns(all_columns, primary_key)

In [0]:
if verbose_mode == 'debug': 
    print(additional_columns)


handle_table_update(
    df_dim_cdc, 
    target_dim_specification, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

dim_site

In [0]:
dim_site = f"""
SELECT id_plant_production_line AS id_plant, name AS plant
FROM {source}.plants_production_lines
WHERE deleted = false
"""

df_dim_site = spark.sql(dim_site)

Import dim_site

In [0]:
current_process="dim_site"

In [0]:
target_dim_site = current_catalog +"."+current_schema+"."+current_process
print(target_dim_site)

In [0]:
all_columns =  df_dim_site.columns
display(all_columns)

In [0]:
# define the primary key 
primary_key = [    
    'id_plant']

additional_columns = get_additional_columns(all_columns, primary_key)

In [0]:
if verbose_mode == 'debug': 
    print(additional_columns)


handle_table_update(
    df_dim_site, 
    target_dim_site, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )